# Using the IPUMS API to Collect Census Extracts for Project Exercises

**MGA 60211 / Advanced Econometrics**


## Note

This code was adapted from the in class IPUMS collection script.

## 1. Setup

First, install the `ipumspy` package (if you haven't already) and import the libraries we need.

In [1]:
# Uncomment the line below if ipumspy is not installed
!pip install ipumspy

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------- ----------------------------- 3.1/12.3 MB 15.3 MB/s eta 0:00:01
   --------------------- ------------------ 6.6/12.3 MB 15.5 MB/s eta 0:00:01
   ---------------------------------- ----- 10.5/12.3 MB 16.4 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 15.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ------ --------------------------------- 1.8/11.0 MB 10.1 MB/s eta 0:00:01
   --------------- ------------------------ 4.2/11.0 MB 10.5 MB/s eta 0:00:01
   --------------------- ------------------ 6.0/11.0 MB 10.0 MB/s eta 0:00:01
   ----------------------------- ---------- 8.1/11.0 MB 9.9 MB/s eta 0:00:01
   -------------------------------------- - 10.5/11.0 MB 9.9 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 9.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   --- --

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.4 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.4.4 which is incompatible.
spyder 5.5.1 requires ipython!=8.17.1,<9.0.0,>=8.13.0; python_version > "3.8", but you have ipython 8.12.3 which is incompatible.
yapf 0.40.2 requires importlib-metadata>=6.6.0, but you have importlib-metadata 4.13.0 which is incompatible.


In [ ]:
import os
from pathlib import Path

from ipumspy import IpumsApiClient, MicrodataExtract, readers
import pandas as pd

# Load API key from environment variable
IPUMS_API_KEY = os.environ.get("IPUMS_API_KEY")
if not IPUMS_API_KEY:
    raise ValueError(
        "IPUMS_API_KEY environment variable not set.\n"
        "Get your key at: https://account.ipums.org/api_keys\n"
        "Then set it: export IPUMS_API_KEY='your-key-here'"
    )

# Initialize the API client
ipums = IpumsApiClient(IPUMS_API_KEY)
print("IPUMS API client initialized.")

IPUMS API client initialized.


## Define an Extract

Gather 1970-2010 data

In [5]:
# Define samples & variables

DECADES = {
    "1970": ["us1970c"],   # 1% Metro — only sample with CNTYGP97
    "1980": ["us1980a"],   # 5% State
    "1990": ["us1990a"],   # 5% State
    "2000": ["us2000a"],   # 5%
    "2010": ["us2010a"],   # ACS
}

# Variables available in ALL samples
BASE_VARS = [
    "YEAR",
    "STATEFIP",
    "PERWT",
    "AGE",
    "GQ",
    "BPLD",     # detailed birthplace -> origin classification
    "EDUCD",    # detailed education -> high/low ed split
    "INCWAGE",  # wage income -> outcome variable
]

# Per-decade variables:
#
#  GEOGRAPHY:
#   1970 → CNTYGP97  (county groups, Metro sample only)
#   1980 → CNTYGP98  (county groups)
#   1990+ → PUMA
#
#  HOURS WORKED:
#   1970 → HRSWORK2  (interval hours last week; UHRSWORK not available in 1970)
#   1980+ → UHRSWORK (usual hours per week)
#
#  WEEKS WORKED:
#   1970 → WKSWORK2  (interval weeks; WKSWORK1 not available in 1970)
#   1980–2000 → WKSWORK1 (exact weeks)
#   2010 ACS → WKSWORK2  (WKSWORK1 not available in 2010 ACS)

DECADE_EXTRA_VARS = {
    "1970": ["CNTYGP97", "HRSWORK2", "WKSWORK2"],
    "1980": ["CNTYGP98", "UHRSWORK", "WKSWORK1"],
    "1990": ["PUMA",     "UHRSWORK", "WKSWORK1"],
    "2000": ["PUMA",     "UHRSWORK", "WKSWORK1"],
    "2010": ["PUMA",     "UHRSWORK", "WKSWORK2"],
}

# Create one extract per decade
extracts = {}
for decade, samples in DECADES.items():
    variables = BASE_VARS + DECADE_EXTRA_VARS[decade]
    extracts[decade] = MicrodataExtract(
        collection="usa",
        description=f"Immigration shift-share: {decade} Census",
        samples=samples,
        variables=variables,
        data_format="fixed_width",
    )

print(f"Defined {len(extracts)} extracts:")
for decade, ext in extracts.items():
    print(f"  {decade}: sample={ext.samples}, {len(ext.variables)} variables")

Defined 5 extracts:
  1970: sample=[Sample(id='us1970c', description='')], 11 variables
  1980: sample=[Sample(id='us1980a', description='')], 11 variables
  1990: sample=[Sample(id='us1990a', description='')], 11 variables
  2000: sample=[Sample(id='us2000a', description='')], 11 variables
  2010: sample=[Sample(id='us2010a', description='')], 11 variables


In [6]:
# Submit all extracts
for decade, ext in extracts.items():
    ipums.submit_extract(ext)
    print(f"{decade}: submitted (ID {ext.extract_id})")

1970: submitted (ID 19)
1980: submitted (ID 20)
1990: submitted (ID 21)
2000: submitted (ID 22)
2010: submitted (ID 23)


In [7]:
# Wait for all extracts to finish
for decade, ext in extracts.items():
    ipums.wait_for_extract(ext, timeout=3600)
    print(f"{decade}: ready")

1970: ready
1980: ready
1990: ready
2000: ready
2010: ready


In [13]:
# dowload as .dta files
download_dir = Path(r"G:\My Drive\Advanced Econometrics for Public Policy and Global Affairs\Advanced Econometrics for Public Policy and Global Affairs Project Excercises\data\raw")
download_dir.mkdir(exist_ok=True)

for decade in extracts.keys():
    data_file = download_dir / f"usa_{decade}.dat"
    ddi_file  = download_dir / f"usa_{decade}.xml"


    ddi = readers.read_ipums_ddi(ddi_file)
    df  = readers.read_microdata(ddi, data_file)

    # Save as .dta
    dta_path = download_dir / f"ipums_usa_{decade}.dta"
    df.to_stata(dta_path, write_index=False, version=118)
    print(f"{decade}: saved → {dta_path.name}")

c:\Users\zpetk\anaconda3\Lib\site-packages\ipumspy\readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


1970: saved → ipums_usa_1970.dta


c:\Users\zpetk\anaconda3\Lib\site-packages\ipumspy\readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


1980: saved → ipums_usa_1980.dta


c:\Users\zpetk\anaconda3\Lib\site-packages\ipumspy\readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


1990: saved → ipums_usa_1990.dta


c:\Users\zpetk\anaconda3\Lib\site-packages\ipumspy\readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


2000: saved → ipums_usa_2000.dta


c:\Users\zpetk\anaconda3\Lib\site-packages\ipumspy\readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


2010: saved → ipums_usa_2010.dta


## AI Documentation

I was running into issues with an error since certain variables did not exist for certain decades when using the sample code from class. I used Claude Sonnet 4.6 to help modify the code from class to select specific variables that vary by decade. I have also attached a log to document this usage. I did not directly use the AI generated code as there were still issues, but I used the code specifically for the Define the Extract section. Here is the link to my AI log: https://claude.ai/share/56d6508b-3f03-44c1-b3cc-4960ad4ba291